# Lab 9: Support Vector Machine (SVM) and Principal Component Analysis (PCA)

## Aim
To implement Support Vector Machine (SVM) for classification and Principal Component Analysis (PCA) for dimensionality reduction, and analyse their effectiveness using real-world datasets.

## Objectives
1. To implement and evaluate the performance of the Support Vector Machine (SVM) classifier using different kernel functions.
2. To compare the performance of SVM models using standard classification metrics.
3. To apply Principal Component Analysis (PCA) for reducing the dimensionality of a high-dimensional dataset.
4. To analyse the variance retained by the principal components and visualize the transformed feature space.
5. To understand the role of supervised learning (SVM) and unsupervised learning (PCA) in machine learning applications.

### Datasets Used
- **Part A:** UCI Breast Cancer Wisconsin (Diagnostic) Dataset — `wdbc.data`
- **Part B:** UCI Wine Dataset — `wine.data`

> Keep `wine.zip` and `breast+cancer+wisconsin+diagnostic.zip` in the same folder as this notebook before running it.


## Setup: Import Libraries and Extract the Uploaded UCI Datasets

This cell imports all required Python libraries and extracts the two UCI ZIP files.

The notebook uses the **actual `wdbc.data` and `wine.data` files**, rather than Scikit-learn's built-in dataset loaders.


In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

sns.set_style("whitegrid")

# Folder where extracted data will be stored
DATA_DIR = "lab9_data"
os.makedirs(DATA_DIR, exist_ok=True)

# Look for ZIP files in the current folder first.
# /mnt/data paths are included so the notebook also runs in the ChatGPT environment.
wine_zip_candidates = [
    "wine.zip",
    "/mnt/data/wine.zip"
]

breast_zip_candidates = [
    "breast+cancer+wisconsin+diagnostic.zip",
    "/mnt/data/breast+cancer+wisconsin+diagnostic.zip"
]

def find_existing_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

wine_zip = find_existing_file(wine_zip_candidates)
breast_zip = find_existing_file(breast_zip_candidates)

if wine_zip is None:
    raise FileNotFoundError("wine.zip was not found. Place it in the same folder as this notebook.")

if breast_zip is None:
    raise FileNotFoundError(
        "breast+cancer+wisconsin+diagnostic.zip was not found. "
        "Place it in the same folder as this notebook."
    )

wine_extract_dir = os.path.join(DATA_DIR, "wine")
breast_extract_dir = os.path.join(DATA_DIR, "breast_cancer")

os.makedirs(wine_extract_dir, exist_ok=True)
os.makedirs(breast_extract_dir, exist_ok=True)

with zipfile.ZipFile(wine_zip, "r") as z:
    z.extractall(wine_extract_dir)

with zipfile.ZipFile(breast_zip, "r") as z:
    z.extractall(breast_extract_dir)

wine_file = os.path.join(wine_extract_dir, "wine.data")
breast_file = os.path.join(breast_extract_dir, "wdbc.data")

print("Wine dataset:", wine_file)
print("Breast Cancer dataset:", breast_file)


# Part A: Support Vector Machine (SVM)

## Aim
To implement the Support Vector Machine (SVM) classifier for a binary classification problem and evaluate its performance.

## Dataset
**UCI Breast Cancer Wisconsin (Diagnostic) Dataset**

The file used is `wdbc.data`. Each row contains:
- ID number
- Diagnosis (`M` = Malignant, `B` = Benign)
- 30 numerical features calculated from digitized images of breast mass nuclei


## Task 1: Load the Dataset and Perform Necessary Preprocessing

### Steps
1. Read `wdbc.data`.
2. Assign meaningful column names.
3. Separate features and target.
4. Check the dataset shape and missing values.
5. Convert diagnosis labels to numerical values:
   - Malignant = 1
   - Benign = 0
6. Inspect the class distribution.

Feature scaling will be performed after the train-test split to prevent data leakage.


In [ ]:
# Column names for the UCI WDBC dataset
feature_names = [
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean",
    "smoothness_mean", "compactness_mean", "concavity_mean",
    "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",

    "radius_se", "texture_se", "perimeter_se", "area_se",
    "smoothness_se", "compactness_se", "concavity_se",
    "concave_points_se", "symmetry_se", "fractal_dimension_se",

    "radius_worst", "texture_worst", "perimeter_worst", "area_worst",
    "smoothness_worst", "compactness_worst", "concavity_worst",
    "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
]

columns = ["id", "diagnosis"] + feature_names

# Load the actual UCI file
df_cancer = pd.read_csv(breast_file, header=None, names=columns)

print("First five rows:")
display(df_cancer.head())

print("\nDataset shape:", df_cancer.shape)
print("\nTotal missing values:", df_cancer.isnull().sum().sum())

print("\nDiagnosis distribution:")
print(df_cancer["diagnosis"].value_counts())

# Convert target labels: M = 1, B = 0
df_cancer["target"] = df_cancer["diagnosis"].map({"M": 1, "B": 0})

# Input features and target
X = df_cancer[feature_names]
y = df_cancer["target"]

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget coding: Malignant = 1, Benign = 0")


In [ ]:
# Visualize class distribution
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df_cancer,
    x="diagnosis",
    order=["B", "M"],
    palette="Set2"
)

plt.title("Breast Cancer Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Number of Samples")
plt.xticks([0, 1], ["Benign", "Malignant"])

plt.show()


## Task 2: Split the Dataset into Training and Testing Sets (80:20)

The dataset is divided into:
- **80% training data**
- **20% testing data**

`random_state=42` makes the split reproducible.

`stratify=y` preserves approximately the same malignant/benign class proportion in both sets.

After splitting, `StandardScaler` is fitted **only on the training data** and then applied to both training and testing data. This avoids data leakage.


In [ ]:
# 80:20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)
print("Training labels  :", y_train.shape)
print("Testing labels   :", y_test.shape)

# Standardize the data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature standardization completed.")


## Task 3: Train an SVM Classifier Using a Linear Kernel and Explore Hyperparameter Tuning

Support Vector Machine finds a separating hyperplane with the maximum possible margin between classes.

### Linear Kernel
A linear kernel is suitable when the classes can be separated reasonably well by a linear boundary.

### Hyperparameter `C`
`C` controls the trade-off between:
- maximizing the margin, and
- penalizing classification errors.

We first train a baseline Linear SVM using `C=1`, then use `GridSearchCV` to test several values of `C`.


In [ ]:
# Baseline Linear SVM
svm_linear = SVC(
    kernel="linear",
    C=1.0
)

svm_linear.fit(X_train_scaled, y_train)

y_pred_linear = svm_linear.predict(X_test_scaled)

print("Baseline Linear SVM trained successfully.")
print("Baseline Accuracy:", round(accuracy_score(y_test, y_pred_linear), 4))


### Hyperparameter Tuning Using GridSearchCV

Five-fold cross-validation is used to evaluate different values of `C`.

The value with the highest mean validation accuracy is selected as the best parameter.


In [ ]:
# Hyperparameter grid
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=SVC(kernel="linear"),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best C:", grid_search.best_params_["C"])
print("Best Cross-Validation Accuracy:", round(grid_search.best_score_, 4))

best_svm = grid_search.best_estimator_
y_pred_tuned = best_svm.predict(X_test_scaled)


In [ ]:
# Display GridSearchCV results
tuning_results = pd.DataFrame(grid_search.cv_results_)

tuning_display = tuning_results[
    ["param_C", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")

display(tuning_display)


## Task 4: Evaluate the Model

The tuned SVM model is evaluated using:

- **Accuracy:** Overall proportion of correctly classified samples.
- **Precision:** Among samples predicted as malignant, how many are actually malignant.
- **Recall:** Among all actual malignant samples, how many are correctly detected.
- **F1 Score:** Harmonic mean of precision and recall.
- **Confusion Matrix:** Shows True Negatives, False Positives, False Negatives and True Positives.

For this notebook, **Malignant = 1** is treated as the positive class.


In [ ]:
# Evaluation metrics for tuned SVM
accuracy = accuracy_score(y_test, y_pred_tuned)
precision = precision_score(y_test, y_pred_tuned)
recall = recall_score(y_test, y_pred_tuned)
f1 = f1_score(y_test, y_pred_tuned)

evaluation_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1]
})

display(evaluation_df)

print("Classification Report:\n")
print(
    classification_report(
        y_test,
        y_pred_tuned,
        target_names=["Benign", "Malignant"]
    )
)


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign", "Malignant"],
    yticklabels=["Benign", "Malignant"]
)

plt.title("Confusion Matrix - Tuned Linear SVM")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.show()


### Additional Kernel Comparison

The objective also mentions evaluating SVM using different kernel functions.

The following kernels are compared:
- Linear
- Polynomial
- Radial Basis Function (RBF)
- Sigmoid

All models use the same standardized training and testing datasets.


In [ ]:
kernels = ["linear", "poly", "rbf", "sigmoid"]

kernel_results = []

for kernel in kernels:
    model = SVC(kernel=kernel)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)

    kernel_results.append({
        "Kernel": kernel,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred)
    })

kernel_results_df = pd.DataFrame(kernel_results)

display(kernel_results_df)


In [ ]:
# Visual comparison of kernel performance
kernel_results_df.set_index("Kernel").plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Comparison of SVM Kernel Performance")
plt.ylabel("Score")
plt.xlabel("Kernel")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc="lower right")

plt.show()


## Task 5: Observations

The Breast Cancer Wisconsin Diagnostic dataset contains 30 numerical features and two classes: malignant and benign.

The features were standardized before SVM training because SVM is sensitive to feature scale.

A Linear SVM was first trained using the default value `C=1`. Hyperparameter tuning was then performed using GridSearchCV to select a suitable value of `C`.

The model was evaluated using accuracy, precision, recall, F1 score and a confusion matrix. Recall for the malignant class is particularly important in this application because a false negative represents a malignant case incorrectly predicted as benign.

Different SVM kernels were also compared. Their performance may differ depending on whether the relationship between the classes is approximately linear or requires a nonlinear decision boundary.

Overall, SVM is a supervised machine learning technique that can provide strong performance on binary classification problems when the features are appropriately scaled.


# Part B: Principal Component Analysis (PCA)

## Aim
To implement Principal Component Analysis (PCA) for dimensionality reduction and analyse the variance retained by the principal components.

## Dataset
**UCI Wine Dataset**

The file used is `wine.data`.

The dataset contains:
- 178 samples
- 13 numerical chemical features
- 3 wine classes


## Task 1: Load the Dataset and Perform Feature Standardization

The Wine dataset contains 13 numerical input features measured on different scales.

PCA is affected by the scale of the variables. Therefore, all 13 features are standardized before applying PCA.

`StandardScaler` transforms each feature to approximately:
- Mean = 0
- Standard deviation = 1


In [ ]:
wine_columns = [
    "class",
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280_od315",
    "proline"
]

# Load the actual UCI wine.data file
df_wine = pd.read_csv(
    wine_file,
    header=None,
    names=wine_columns
)

print("First five rows:")
display(df_wine.head())

print("\nDataset shape:", df_wine.shape)
print("Total missing values:", df_wine.isnull().sum().sum())

print("\nClass distribution:")
print(df_wine["class"].value_counts().sort_index())

# Separate features and class
X_wine = df_wine.drop(columns="class")
y_wine = df_wine["class"]

# Standardize all 13 numerical features
wine_scaler = StandardScaler()
X_wine_scaled = wine_scaler.fit_transform(X_wine)

print("\nOriginal feature count:", X_wine.shape[1])
print("Standardized data shape:", X_wine_scaled.shape)


## Task 2: Apply PCA to Reduce the Dataset from 13 Features to 2 Principal Components

PCA transforms the original features into new uncorrelated variables called **principal components**.

- **PC1** captures the maximum possible variance.
- **PC2** captures the next highest variance while remaining orthogonal to PC1.

For visualization, the 13 original features are reduced to two principal components.


In [ ]:
# PCA with 2 principal components
pca_2 = PCA(n_components=2)

X_pca_2 = pca_2.fit_transform(X_wine_scaled)

pca_df = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)

pca_df["Class"] = y_wine.values

print("Original dataset shape:", X_wine.shape)
print("PCA transformed feature shape:", X_pca_2.shape)

display(pca_df.head())


## Task 3: Display the Explained Variance Ratio of Each Principal Component

The explained variance ratio measures how much of the total variance in the original data is captured by each principal component.

A larger explained variance ratio means that the component preserves more information from the original feature space.


In [ ]:
explained_variance_2 = pca_2.explained_variance_ratio_

variance_2_df = pd.DataFrame({
    "Principal Component": ["PC1", "PC2"],
    "Explained Variance Ratio": explained_variance_2,
    "Explained Variance (%)": explained_variance_2 * 100
})

display(variance_2_df)

print(
    "Total variance retained by PC1 and PC2:",
    f"{explained_variance_2.sum() * 100:.2f}%"
)


## Task 4: Calculate Cumulative Explained Variance and Find the Minimum Components for 95% Variance

PCA is applied again without limiting the number of components.

The explained variance ratios are accumulated to determine how many principal components are required to preserve at least **95% of the total variance**.


In [ ]:
# PCA using all available components
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_wine_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

variance_df = pd.DataFrame({
    "Principal Component": [f"PC{i+1}" for i in range(len(explained_variance))],
    "Explained Variance Ratio": explained_variance,
    "Cumulative Explained Variance": cumulative_variance
})

display(variance_df)

# Minimum number of components needed for at least 95%
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print(
    "Minimum number of principal components needed to retain at least 95% variance:",
    n_components_95
)

print(
    "Variance retained:",
    f"{cumulative_variance[n_components_95 - 1] * 100:.2f}%"
)


In [ ]:
# Plot cumulative explained variance
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)

plt.axhline(
    y=0.95,
    color="red",
    linestyle="--",
    label="95% Variance"
)

plt.axvline(
    x=n_components_95,
    color="green",
    linestyle="--",
    label=f"{n_components_95} Components"
)

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance - Wine Dataset")
plt.xticks(range(1, 14))
plt.ylim(0, 1.05)
plt.legend()

plt.show()


## Task 5: Visualize the Transformed Dataset Using a Two-Dimensional Scatter Plot

The PCA-transformed dataset is visualized using PC1 and PC2 as the two axes.

Each point represents one wine sample, and the color indicates the wine class.

This plot helps us inspect how much class structure is visible after reducing 13 dimensions to only 2 dimensions.


In [ ]:
plot_df = pd.DataFrame({
    "PC1": X_pca_2[:, 0],
    "PC2": X_pca_2[:, 1],
    "Class": y_wine.astype(str).values
})

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="Class",
    palette="Set1",
    s=80,
    alpha=0.8
)

plt.title("Wine Dataset after PCA - First Two Principal Components")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Wine Class")

plt.show()


## Task 6: Compare the Original and Transformed Datasets

The original dataset contains 13 features, while the visualization-oriented PCA representation contains only 2 principal components.

### Comparison Criteria
- Number of features
- Information retained
- Computational efficiency


In [ ]:
information_retained_2 = pca_2.explained_variance_ratio_.sum() * 100

comparison_df = pd.DataFrame({
    "Aspect": [
        "Number of Features",
        "Information Retained",
        "Computational Efficiency",
        "Interpretability"
    ],
    "Original Dataset": [
        "13",
        "100% of original feature representation",
        "Lower efficiency because more dimensions are processed",
        "Original features have direct chemical meaning"
    ],
    "PCA Dataset (2 PCs)": [
        "2",
        f"{information_retained_2:.2f}% of total variance",
        "Higher efficiency because fewer dimensions are processed",
        "PCs are combinations of the original features"
    ]
})

display(comparison_df)


## Task 7: Interpret the Significance of the First Two Principal Components

PC1 and PC2 are not individual original attributes. Each principal component is a weighted linear combination of all 13 standardized wine features.

- **PC1** represents the direction with the highest variance in the dataset.
- **PC2** represents the second-highest variance and is orthogonal to PC1.

To understand which original features influence PC1 and PC2 the most, PCA **loadings** are examined.

Large absolute loading values indicate stronger contributions to a principal component.


In [ ]:
# PCA loadings
loadings = pd.DataFrame(
    pca_2.components_.T,
    columns=["PC1", "PC2"],
    index=X_wine.columns
)

print("PCA Loadings:")
display(loadings)

print("\nTop 5 contributors to PC1:")
display(
    loadings["PC1"]
    .abs()
    .sort_values(ascending=False)
    .head(5)
)

print("\nTop 5 contributors to PC2:")
display(
    loadings["PC2"]
    .abs()
    .sort_values(ascending=False)
    .head(5)
)


In [ ]:
# Loading heatmap
plt.figure(figsize=(8, 8))

sns.heatmap(
    loadings,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Feature Loadings for PC1 and PC2")
plt.xlabel("Principal Components")
plt.ylabel("Original Wine Features")

plt.show()


### Interpretation

The variables having the largest absolute loadings on PC1 contribute most strongly to the primary direction of variation in the Wine dataset.

Similarly, the variables with the largest absolute loadings on PC2 contribute most strongly to the second direction of variation.

The exact important features are displayed by the previous code cell. This data-driven interpretation is preferable to assuming which features are important before running PCA.

If the scatter plot shows visible grouping of the three wine classes, it indicates that PC1 and PC2 preserve meaningful structural information even though the dataset has been reduced from 13 dimensions to only 2.


## Task 8: Advantages, Limitations, and Applications of PCA

### Advantages
1. Reduces the dimensionality of high-dimensional datasets.
2. Can reduce training time and memory usage.
3. Removes redundancy caused by correlated features.
4. Helps visualize high-dimensional datasets in two or three dimensions.
5. Can reduce noise by discarding low-variance components.
6. Can reduce the effect of the curse of dimensionality.

### Limitations
1. Principal components may be difficult to interpret.
2. PCA is primarily a linear dimensionality-reduction method.
3. Some information is lost when components are removed.
4. PCA is sensitive to feature scaling.
5. PCA can also be sensitive to strong outliers.
6. High variance is assumed to represent important information, which may not always be true.

### Applications
- Data visualization
- Image compression
- Face recognition
- Pattern recognition
- Feature extraction
- Signal processing
- Bioinformatics
- Financial data analysis
- Noise reduction
- Preprocessing before machine learning model training


# Final Comparison: SVM vs PCA

| Aspect | SVM | PCA |
|---|---|---|
| Learning Type | Supervised | Unsupervised |
| Requires Labels | Yes | No |
| Main Purpose | Classification / Regression | Dimensionality Reduction |
| Main Idea | Finds a maximum-margin decision boundary | Finds directions of maximum variance |
| Output | Predicted class/value | Principal components |
| Typical Use | Classification problems | Feature reduction, visualization, preprocessing |

SVM and PCA can also be used together. PCA can first reduce the number of features, after which SVM can perform classification on the lower-dimensional representation.


# Conclusion

In this laboratory experiment, Support Vector Machine and Principal Component Analysis were implemented using the actual UCI dataset files supplied for the lab.

For Part A, the Breast Cancer Wisconsin Diagnostic dataset was loaded from `wdbc.data`. After preprocessing, the data was divided into 80% training data and 20% testing data. The features were standardized, a Linear SVM was trained, and the `C` hyperparameter was tuned using GridSearchCV. The model was evaluated using accuracy, precision, recall, F1 score and a confusion matrix. Multiple SVM kernels were also compared.

For Part B, the Wine dataset was loaded from `wine.data`. Its 13 numerical features were standardized before PCA was applied. The dataset was reduced to two principal components for visualization, explained variance was calculated, and cumulative explained variance was used to determine the minimum number of components needed to retain at least 95% of the total variance. Feature loadings were also examined to interpret PC1 and PC2.

The experiment demonstrates that SVM is a supervised learning technique used for prediction, while PCA is an unsupervised technique used mainly for dimensionality reduction and feature transformation.
